# Tải Kvasir-SEG và CVC-ClinicDB

Chọn kernel Python của repo, cài `requirements.txt` theo README, rồi **Run All** từ trên xuống. Không cần GPU hoặc API key. Notebook tự tìm gốc repo kể cả khi kernel mở tại `src/datasets/`.

Kết quả: `data/Kvasir-SEG/{images,masks}` có **1.000 cặp JPG** và `data/CVC-ClinicDB/{images,masks}` có **612 cặp PNG**. Giữ nguyên byte ảnh/mask, không resize hay sửa nhãn. ZIP/cache Hugging Face và manifest nằm trong `data/`, được `.gitignore` bỏ qua. Chạy lại sẽ kiểm tra, bỏ qua file khớp và bổ sung file thiếu; file xung đột được báo lỗi thay vì ghi đè.

- **Kvasir-SEG:** [Simula, nguồn gốc và điều kiện sử dụng](https://datasets.simula.no/kvasir-seg/).
- **CVC-ClinicDB:** dùng `snapshot_download(repo_id='berkaytrhn/cvc-clinicdb', repo_type='dataset', ...)` từ [đúng repo Hugging Face của nhóm](https://huggingface.co/datasets/berkaytrhn/cvc-clinicdb), cố định revision `07f1aae53befabc313f9435c8b1b9565642a36c8`. [Nguồn gốc và điều kiện sử dụng](https://polyp.grand-challenge.org/CVCClinicDB/).

Lệnh `snapshot_download` thông thường giữ cây `CVC-ClinicDB/Original/` và `CVC-ClinicDB/Ground Truth/` bên trong thư mục đích. Notebook tải vào cache rồi sao chép nguyên byte sang `images/` và `masks/`, tránh lồng thêm thư mục trong dữ liệu dùng huấn luyện. Metadata CSV được giữ trong cache.

Dữ liệu dành cho nghiên cứu/giáo dục; trích dẫn **Jha et al. (2020)** và **Bernal et al. (2015)** như README. Notebook không tạo split hoặc huấn luyện mô hình.

In [ ]:
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import ssl
import tempfile
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import zipfile

from PIL import Image
from huggingface_hub import HfApi, snapshot_download
from tqdm.auto import tqdm
from IPython.display import display


## 1. Đường dẫn và nguồn tải

Thông thường giữ `PROJECT_ROOT_OVERRIDE = None`. Nếu kernel chạy ngoài cây thư mục repo, điền đường dẫn repo ở biến này. Kvasir-SEG dùng SHA-256 ZIP cố định; CVC-ClinicDB dùng revision Hugging Face cố định và SHA-256 từng PNG từ metadata LFS của revision đó. Không tự đổi sang nguồn khác khi có lỗi tải.

In [ ]:
PROJECT_ROOT_OVERRIDE = None  # Ví dụ: Path(r'D:\duong-dan\Medical-Image-Segmentation')

def find_project_root(override=None):
    candidates = [Path(override).expanduser().resolve()] if override else [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / 'src' / 'datasets').is_dir() and (candidate / 'README.md').is_file() and (candidate / '.gitignore').is_file():
            return candidate.resolve()
    raise FileNotFoundError('Không tìm thấy gốc repo. Điền PROJECT_ROOT_OVERRIDE rồi chạy lại từ cell này.')

PROJECT_ROOT = find_project_root(PROJECT_ROOT_OVERRIDE)
DATA_DIR = PROJECT_ROOT / 'data'
DOWNLOAD_DIR = DATA_DIR / '_downloads'
for directory in (DATA_DIR, DOWNLOAD_DIR, PROJECT_ROOT / 'checkpoints', PROJECT_ROOT / 'logs'):
    directory.mkdir(parents=True, exist_ok=True)

DATASETS = [
    {
        'name': 'Kvasir-SEG',
        'download_type': 'zip',
        'url': 'https://datasets.simula.no/downloads/kvasir-seg.zip',
        'archive_name': 'kvasir-seg.zip',
        'archive_sha256': '03b30e21d584e04facf49397a2576738fd626815771afbbf788f74a7153478f7',
        'expected_pairs': 1000,
        'extension': '.jpg',
        'image_suffix': ('images',),
        'mask_suffix': ('masks',),
        'expected_size': None,
        'source_page': 'https://datasets.simula.no/kvasir-seg/',
        'role': 'train/validation/test nội bộ; split chưa được tạo',
    },
    {
        'name': 'CVC-ClinicDB',
        'download_type': 'huggingface',
        'repo_id': 'berkaytrhn/cvc-clinicdb',
        'revision': '07f1aae53befabc313f9435c8b1b9565642a36c8',
        'url': 'https://huggingface.co/datasets/berkaytrhn/cvc-clinicdb',
        'expected_pairs': 612,
        'extension': '.png',
        'source_directories': {'images': 'CVC-ClinicDB/Original', 'masks': 'CVC-ClinicDB/Ground Truth'},
        'expected_size': (384, 288),  # width, height
        'source_page': 'https://polyp.grand-challenge.org/CVCClinicDB/',
        'mirror_page': 'https://huggingface.co/datasets/berkaytrhn/cvc-clinicdb',
        'role': 'external test; không dùng tinh chỉnh mô hình',
    },
]

print('Project root:', PROJECT_ROOT)
print('Data directory:', DATA_DIR)


## 2. Hàm tải và kiểm tra ZIP Kvasir-SEG

Tải theo từng khối vào file `.part`, chỉ đổi tên khi ZIP có SHA-256 đúng. Có tối đa 3 lần thử với lỗi kết nối tạm thời. HTTPS dùng kho chứng chỉ mặc định của Python/hệ điều hành; không tắt xác minh TLS. File cache sai checksum được báo lỗi để kiểm tra thủ công.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def verify_archive(path, expected_sha256):
    actual = sha256_file(path)
    if actual != expected_sha256:
        raise ValueError(f'Checksum ZIP không khớp: {path}\nKỳ vọng: {expected_sha256}\nThực tế: {actual}\nKiểm tra nguồn; nếu cache hỏng, di chuyển riêng ZIP này ra ngoài cache rồi chạy lại.')
    if not zipfile.is_zipfile(path):
        raise ValueError(f'File không phải ZIP hợp lệ: {path}')

def download_archive(spec):
    destination = DOWNLOAD_DIR / spec['archive_name']
    if destination.exists():
        verify_archive(destination, spec['archive_sha256'])
        print(f"Dùng ZIP cache: {destination.name}")
        return destination

    partial = destination.with_suffix(destination.suffix + '.part')
    for attempt in range(1, 4):
        try:
            request = Request(spec['url'], headers={'User-Agent': 'Medical-Image-Segmentation/1.0'})
            with urlopen(request, timeout=60) as response, partial.open('wb') as handle:
                total = int(response.headers.get('Content-Length') or 0) or None
                downloaded = 0
                with tqdm(total=total, unit='B', unit_scale=True, desc=spec['name']) as progress:
                    while True:
                        chunk = response.read(1024 * 1024)
                        if not chunk:
                            break
                        handle.write(chunk)
                        downloaded += len(chunk)
                        progress.update(len(chunk))
                if total is not None and downloaded != total:
                    raise OSError(f'Tải thiếu dữ liệu: {downloaded}/{total} byte')
            if not zipfile.is_zipfile(partial):
                raise ValueError('Máy chủ không trả ZIP (có thể là trang HTML/đăng nhập). Kiểm tra URL và kết nối, không nhập API key vào notebook.')
            verify_archive(partial, spec['archive_sha256'])
            partial.replace(destination)
            return destination
        except (HTTPError, URLError, TimeoutError, OSError, ValueError) as exc:
            partial.unlink(missing_ok=True)  # Chỉ xóa file tạm do lần tải này tạo.
            ssl_error = isinstance(exc, ssl.SSLError) or isinstance(getattr(exc, 'reason', None), ssl.SSLError)
            if ssl_error:
                raise RuntimeError('Không xác minh được chứng chỉ HTTPS. Cập nhật Python/chứng chỉ của môi trường; không dùng verify=False. Chi tiết lỗi gốc ở dưới.') from exc
            if isinstance(exc, ValueError) or (isinstance(exc, HTTPError) and exc.code not in {408, 429, 500, 502, 503, 504}):
                raise RuntimeError(f"Không tải được {spec['name']}: {exc}") from exc
            if attempt == 3:
                raise RuntimeError(f"Không tải được {spec['name']} sau 3 lần thử. Kiểm tra mạng và chạy lại.") from exc
            print(f'Lần tải {attempt}/3 thất bại: {exc}. Thử lại...')


## 3. Lấy đúng ảnh–mask, giữ nguyên dữ liệu

Kvasir-SEG: đọc từng thành viên ZIP đã chọn, chỉ dùng tên file an toàn và thư mục cố định. CVC-ClinicDB: lấy đúng PNG từ snapshot Hugging Face đã khóa revision. Đối chiếu file đã có trước khi thêm file thiếu; không tự thay nội dung file đã tồn tại.

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}

def archive_members(archive, spec):
    groups = {'images': {}, 'masks': {}}
    for member in archive.infolist():
        path = PurePosixPath(member.filename)
        if member.is_dir() or path.suffix.lower() != spec['extension'] or '__MACOSX' in path.parts:
            continue
        if path.is_absolute() or '..' in path.parts or chr(92) in member.filename or ':' in member.filename:
            raise ValueError(f'Tên thành viên ZIP không an toàn: {member.filename}')
        parent = tuple(part.casefold() for part in path.parts[:-1])
        for group, suffix in [('images', spec['image_suffix']), ('masks', spec['mask_suffix'])]:
            if parent[-len(suffix):] == suffix:
                if path.name in groups[group]:
                    raise ValueError(f'Trùng tên trong ZIP: {group}/{path.name}')
                groups[group][path.name] = member

    images, masks = groups['images'], groups['masks']
    expected = spec['expected_pairs']
    if len(images) != expected or len(masks) != expected or set(images) != set(masks):
        raise ValueError(f"{spec['name']}: ZIP phải có {expected} cặp khớp tên; thực tế {len(images)} ảnh, {len(masks)} mask.")
    return groups

def write_new_file(path, payload):
    # Chế độ xb bảo đảm không ghi đè kể cả khi file vừa được tiến trình khác tạo.
    with path.open('xb') as handle:
        try:
            handle.write(payload)
        except BaseException:
            handle.close()
            path.unlink(missing_ok=True)
            raise

def prepare_zip_dataset(spec):
    zip_path = download_archive(spec)
    target = DATA_DIR / spec['name']
    with zipfile.ZipFile(zip_path) as archive:
        groups = archive_members(archive, spec)
        hashes = {}
        pending = []
        conflicts = []
        for group, members in groups.items():
            directory = target / group
            directory.mkdir(parents=True, exist_ok=True)
            unexpected = [p.name for p in directory.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS and p.name not in members]
            if unexpected:
                conflicts.append(f'{directory}: file ảnh ngoài danh sách nguồn: {unexpected[:5]}')
            for filename, member in tqdm(sorted(members.items()), desc=f"Đối chiếu {spec['name']}/{group}", leave=False):
                payload = archive.read(member)  # zipfile kiểm tra CRC khi đọc.
                digest = hashlib.sha256(payload).hexdigest()
                relative = f'{group}/{filename}'
                hashes[relative] = digest
                destination = directory / filename
                if destination.exists():
                    if not destination.is_file() or sha256_file(destination) != digest:
                        conflicts.append(str(destination))
                else:
                    pending.append((member, destination))
        if conflicts:
            raise RuntimeError('Phát hiện dữ liệu khác nguồn. Không ghi đè/xóa file; kiểm tra các đường dẫn sau:\n' + '\n'.join(conflicts[:10]))
        for member, destination in tqdm(pending, desc=f"Tạo file {spec['name']}", leave=False):
            write_new_file(destination, archive.read(member))
    print(f"{spec['name']}: thêm {len(pending)} file, giữ {len(hashes) - len(pending)} file khớp nguồn.")
    return hashes

def prepare_huggingface_dataset(spec):
    # Metadata được đọc ở revision cố định, không đọc nhánh main có thể thay đổi.
    api = HfApi(token=False)
    entries = api.list_repo_tree(repo_id=spec['repo_id'], repo_type='dataset', revision=spec['revision'], path_in_repo='CVC-ClinicDB', recursive=True)
    sources = {}
    expected_names = {f'{index}.png' for index in range(1, spec['expected_pairs'] + 1)}
    source_groups = {directory: group for group, directory in spec['source_directories'].items()}
    for entry in entries:
        path = PurePosixPath(entry.path)
        if str(path.parent) not in source_groups or path.suffix.lower() != '.png':
            continue
        if path.is_absolute() or '..' in path.parts or chr(92) in entry.path or ':' in entry.path:
            raise ValueError(f'Đường dẫn Hugging Face không an toàn: {entry.path}')
        lfs = getattr(entry, 'lfs', None)
        digest = getattr(lfs, 'sha256', None)
        if not digest or len(digest) != 64:
            raise ValueError(f'Thiếu SHA-256 LFS tại revision đã chốt: {entry.path}')
        relative = f'{source_groups[str(path.parent)]}/{path.name}'
        if relative in sources:
            raise ValueError(f'Trùng file trong nguồn Hugging Face: {relative}')
        sources[relative] = (entry.path, digest)
    for group in ('images', 'masks'):
        names = {relative.split('/', 1)[1] for relative in sources if relative.startswith(group + '/')}
        if names != expected_names:
            raise ValueError(f"{spec['name']}: nguồn Hugging Face phải có đủ 612 tên PNG cho {group}.")

    target = DATA_DIR / spec['name']
    conflicts = []
    pending = []
    for group in ('images', 'masks'):
        directory = target / group
        directory.mkdir(parents=True, exist_ok=True)
        unexpected = [p.name for p in directory.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS and p.name not in expected_names]
        if unexpected:
            conflicts.append(f'{directory}: file ảnh ngoài danh sách nguồn: {unexpected[:5]}')
    for relative, (remote_path, digest) in tqdm(sorted(sources.items()), desc='Đối chiếu CVC-ClinicDB/Hugging Face', leave=False):
        destination = target / relative
        if destination.exists():
            if not destination.is_file() or sha256_file(destination) != digest:
                conflicts.append(str(destination))
        else:
            pending.append((relative, remote_path, digest))
    if conflicts:
        raise RuntimeError('Phát hiện dữ liệu khác revision Hugging Face. Không ghi đè/xóa file:\n' + '\n'.join(conflicts[:10]))
    if pending:
        # Giữ đường dẫn cache ngắn để tương thích giới hạn đường dẫn Windows.
        cache_dir = DOWNLOAD_DIR / 'clinicdb-hf'
        snapshot_dir = Path(snapshot_download(
            repo_id=spec['repo_id'], repo_type='dataset', revision=spec['revision'],
            local_dir=str(cache_dir), token=False, max_workers=4,
            allow_patterns=[remote_path for _, remote_path, _ in pending] + ['metadata.csv', 'class_dict.csv', 'README.md'],
        ))
        # Kiểm tra hết file tải về trước khi sao chép vào dữ liệu dùng huấn luyện.
        for relative, remote_path, digest in pending:
            source = snapshot_dir / remote_path
            if not source.is_file() or sha256_file(source) != digest:
                raise ValueError(f'File cache Hugging Face sai SHA-256: {source}. Kiểm tra/di chuyển riêng file hỏng rồi chạy lại.')
        for relative, remote_path, digest in tqdm(pending, desc='Tạo file CVC-ClinicDB', leave=False):
            write_new_file(target / relative, (snapshot_dir / remote_path).read_bytes())
    print(f"CVC-ClinicDB: thêm {len(pending)} file, giữ {len(sources) - len(pending)} file khớp revision Hugging Face.")
    return {relative: digest for relative, (_, digest) in sources.items()}

def prepare_dataset(spec):
    if spec['download_type'] == 'zip':
        return prepare_zip_dataset(spec)
    if spec['download_type'] == 'huggingface':
        return prepare_huggingface_dataset(spec)
    raise ValueError(f"Nguồn tải chưa hỗ trợ: {spec['download_type']}")


## 4. Kiểm tra toàn bộ cặp ảnh–mask và ghi manifest

Kiểm tra ảnh đọc được, định dạng đúng, ảnh–mask cùng kích thước, mask có cả vùng nền và polyp khi dùng `grayscale > 127`. Phép nhị phân hóa chỉ để kiểm tra/hiển thị, không ghi lại mask nguồn. Manifest ghi thời điểm, nguồn và SHA-256 từng file để truy xuất bản dữ liệu.

In [ ]:
def validate_dataset(spec, hashes):
    folder = DATA_DIR / spec['name']
    names = sorted(key.split('/', 1)[1] for key in hashes if key.startswith('images/'))
    expected = spec['expected_pairs']
    for group in ('images', 'masks'):
        files = [p.name for p in (folder / group).iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
        if len(files) != expected or set(files) != set(names):
            raise ValueError(f"{spec['name']}/{group}: không đủ {expected} file khớp nguồn.")

    sizes = Counter()
    non_binary_masks = 0
    expected_format = 'JPEG' if spec['extension'] == '.jpg' else 'PNG'
    for filename in tqdm(names, desc=f"Kiểm tra {spec['name']}", leave=False):
        image_path, mask_path = folder / 'images' / filename, folder / 'masks' / filename
        with Image.open(image_path) as image, Image.open(mask_path) as mask:
            image.load()
            mask.load()
            if image.format != expected_format or mask.format != expected_format:
                raise ValueError(f'Định dạng không khớp phần mở rộng: {filename}')
            if image.size != mask.size:
                raise ValueError(f'Kích thước ảnh–mask không khớp: {filename}')
            if spec['expected_size'] and image.size != spec['expected_size']:
                raise ValueError(f'CVC-ClinicDB sai kích thước: {filename}, {image.size}')
            sizes[f'{image.width}x{image.height}'] += 1
            histogram = mask.convert('L').histogram()
            foreground = sum(histogram[128:])
            if foreground == 0 or foreground == image.width * image.height:
                raise ValueError(f'Mask không có đủ nền và polyp: {filename}')
            non_binary_masks += int(any(histogram[1:255]))
    return {
        'pairs': len(names),
        'extension': spec['extension'],
        'size_counts': dict(sorted(sizes.items())),
        'masks_with_intermediate_gray_values': non_binary_masks,
    }

def write_manifest(path, manifest):
    # Ghi thay thế nguyên tử chỉ cho manifest được notebook quản lý.
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', dir=path.parent, suffix='.tmp', delete=False) as handle:
            temporary_path = Path(handle.name)
            json.dump(manifest, handle, ensure_ascii=False, indent=2)
            handle.write('\n')
        os.replace(temporary_path, path)
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)


In [ ]:
manifest = {
    'schema_version': 2,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'mask_check_threshold': 127,
    'datasets': {},
}
summaries = {}
for spec in DATASETS:
    file_hashes = prepare_dataset(spec)
    summary = validate_dataset(spec, file_hashes)
    summaries[spec['name']] = summary
    manifest['datasets'][spec['name']] = {
        'download_url': spec['url'],
        'download_type': spec['download_type'],
        'source_page': spec['source_page'],
        'mirror_page': spec.get('mirror_page'),
        'archive_sha256': spec.get('archive_sha256'),
        'repo_id': spec.get('repo_id'),
        'revision': spec.get('revision'),
        'role': spec['role'],
        'summary': summary,
        'files_sha256': file_hashes,
    }

manifest_path = DATA_DIR / 'dataset_manifest.json'
write_manifest(manifest_path, manifest)
print('\nHoàn tất. Manifest:', manifest_path)
print(f"{'Dataset':<18} {'Ảnh':>6} {'Mask':>6} {'Định dạng':>10}")
for name, summary in summaries.items():
    print(f"{name:<18} {summary['pairs']:>6} {summary['pairs']:>6} {summary['extension']:>10}")
    print('  Mask có mức xám trung gian:', summary['masks_with_intermediate_gray_values'])
print('Dữ liệu sẵn sàng. Chưa tạo split và chưa huấn luyện mô hình.')


## 5. Xem mẫu ảnh, mask và overlay

Mỗi hàng: ảnh RGB → mask hiển thị sau ngưỡng 127 → overlay vùng polyp. Chỉ hiển thị trong notebook; không ghi ảnh mới vào dataset.

In [ ]:
for spec in DATASETS:
    folder = DATA_DIR / spec['name']
    image_path = sorted((folder / 'images').glob('*' + spec['extension']))[0]
    mask_path = folder / 'masks' / image_path.name
    with Image.open(image_path) as source_image, Image.open(mask_path) as source_mask:
        image = source_image.convert('RGB')
        mask = source_mask.convert('L').point(lambda value: 255 if value > 127 else 0)
        red = Image.new('RGB', image.size, (255, 0, 0))
        overlay = Image.composite(Image.blend(image, red, 0.4), image, mask)
        panels = [image, mask.convert('RGB'), overlay]
        canvas = Image.new('RGB', (320 * 3, 280), 'white')
        for index, panel in enumerate(panels):
            panel.thumbnail((320, 280))
            canvas.paste(panel, (index * 320 + (320 - panel.width) // 2, (280 - panel.height) // 2))
        print(f"{spec['name']}: {image_path.name} | Ảnh RGB — mask — overlay")
        display(canvas)
